In [1]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_nuclei_labels_output_dir,
    load_precomputed_nuclei_labels_if_available,
)
from utils.segmentation import predict_nuclei_labels

from scipy.ndimage import distance_transform_edt
from scipy.ndimage import binary_closing, binary_fill_holes
from skimage.morphology import ball

import tifffile
import napari



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2026-04-16 11:32:38,480 [INFO] WRITING LOG OUTPUT TO C:\Users\adiez_cmic\.cellpose\run.log
2026-04-16 11:32:38,480 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.10.20 
torch version:  	2.5.0
2026-04-16 11:32:38,598 [INFO] ** TORCH CUDA version installed and working. **
2026-04-16 11:32:38,598 [INFO] ** TORCH CUDA version installed and working. **
2026-04-16 11:32:38,598 [INFO] >>>> using GPU (CUDA)
2026-04-16 11:32:39,575 [INFO] >>>> loading model C:\Users\adiez_cmic\.cellpose\models\cpsam


In [2]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 0

In [3]:
# Iterate through the .lif files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [4]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (115, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (116, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (135, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (107, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (118, 4, 256, 1024)


In [5]:
# Initialize Napari Viewer
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

2026-04-16 11:32:42,847 [INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


[<Image layer 'edCerulean_CTRL' at 0x1e884b1c610>,
 <Image layer 'edCitrine_FRET' at 0x1e7b37f4940>,
 <Image layer 'edCitrine_CTRL' at 0x1e7b3924880>,
 <Image layer 'brightfield' at 0x1e7b3972650>]

In [6]:
from skimage.filters import difference_of_gaussians

In [7]:
# Create new input array for ConvPaint segmentation ("negative" of brightfield + edCitrine_CTRL) with 1 to 99 percentile normalization

import numpy as np

def normalize_percentile(image, pmin=1, pmax=99):
    vmin, vmax = np.percentile(image, (pmin, pmax))
    image = np.clip(image, vmin, vmax)
    if vmax > vmin:
        image = (image - vmin) / (vmax - vmin)
    else:
        image = image * 0  # avoid division by zero; returns zeros
    return image

brightfield_norm = normalize_percentile(lif_image[3])
edcitrine_ctrl_norm = normalize_percentile(lif_image[2])

bf_dog = difference_of_gaussians(brightfield_norm, low_sigma=1, high_sigma=2) # Remove brightfield haze
bf_inv = 1 - bf_dog

convpaint_input = ((1 - brightfield_norm) + edcitrine_ctrl_norm) / 2

# Add convpaint input to napari viewer
viewer.add_image(convpaint_input, 
                name="convpaint_input",
                colormap="gray",
                blending="additive")
           

<Image layer 'convpaint_input' at 0x1e7b3dfb520>

In [ ]:
# Optional: PanSeg-like tiled + halo UNet inference (standalone)
# Expects model_dir to contain:
#   - config_train.yml
#   - best_checkpoint.pytorch

import torch
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))
from utils.inference import predict_tiled_unet

# Example input from this notebook:
# bf_inv has shape (Z, Y, X)
raw_for_unet = bf_inv

model_dir = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

root_pmaps = predict_tiled_unet(
    raw=raw_for_unet,
    input_layout="ZYX",
    model_dir=model_dir,
    patch=(80, 160, 160),
    patch_halo=(8, 16, 16),
    stride_ratio=0.75,
    batch_size=1,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_amp=True,
)

# root_pmaps: (C_out, Z, Y, X)
viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="magenta", blending="additive")

c:\Users\adiez_cmic\github_repos\saramorg_fret_nroot\notebooks\..\src\utils\inference.py:163: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path, 

<Image layer 'root_unet_pmap' at 0x1e7b21a7dc0>

In [9]:
raise RuntimeError("Stopping execution here (intentional break). Remove or comment out to continue running all cells.")

RuntimeError: Stopping execution here (intentional break). Remove or comment out to continue running all cells.

In [ ]:
import pyclesperanto_prototype as cle

root_labels = cle.voronoi_otsu_labeling(convpaint_input, spot_sigma=100, outline_sigma=20)

viewer.add_labels(root_labels, name="voronoi_otsu_labeling")

In [10]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_nuclei_labels_output_dir(RAW_DATA_DIRECTORY, lif_container_id)
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_nuclei_labels_if_available(nuclei_labels_dir, lif_image_name)

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Add the prediction to the napari viewer
viewer.add_labels(nuclei_labels)

Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_mock 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
Predictions already calculated for: Col0 1 ...loading


<Labels layer 'nuclei_labels' at 0x1e7b45d6b90>

In [11]:
root_boundary_mask = root_pmaps[0] > 0.8
viewer.add_image(root_boundary_mask, name="root_boundary_mask", colormap="gray", blending="additive", opacity=0.5)

<Image layer 'root_boundary_mask' at 0x1e7b4b5fe20>

In [15]:
from scipy.ndimage import binary_fill_holes

# Apply binary closing

# Fill holes in the mask
root_mask = binary_fill_holes(root_boundary_mask, structure=np.ones((5,5,5)))
viewer.add_image(root_mask, name="root_mask", colormap="gray", blending="additive", opacity=0.5)

<Image layer 'root_mask [1]' at 0x1e7b52bfac0>

In [ ]:
import numpy as np
from scipy.ndimage import binary_fill_holes
from skimage.morphology import (
    remove_small_holes,
    binary_closing,
    ball,
    disk
)
from skimage.measure import label

# Ensure boolean
mask = root_boundary_mask.astype(bool)

# --------------------------------------------------
# 1. Pure 3D hole filling (often fails if boundaries leak in Z)
# --------------------------------------------------
filled_3d = binary_fill_holes(mask)

# --------------------------------------------------
# 2. 3D closing + fill (fix small gaps in all directions)
# --------------------------------------------------
closed_3d = binary_closing(mask, footprint=ball(2))
filled_closed_3d = binary_fill_holes(closed_3d)

# --------------------------------------------------
# 3. 3D remove small holes (robust alternative)
# --------------------------------------------------
holes_removed_3d = remove_small_holes(mask, area_threshold=5000)

# --------------------------------------------------
# 4. Slice-wise (2D) hole filling (VERY useful here)
# --------------------------------------------------
filled_2d = np.zeros_like(mask)

for z in range(mask.shape[0]):
    filled_2d[z] = binary_fill_holes(mask[z])

# --------------------------------------------------
# 5. Slice-wise closing + filling (best for broken contours)
# --------------------------------------------------
filled_2d_closed = np.zeros_like(mask)

for z in range(mask.shape[0]):
    closed = binary_closing(mask[z], footprint=disk(2))
    filled_2d_closed[z] = binary_fill_holes(closed)

# --------------------------------------------------
# 6. Keep largest connected component (3D cleanup)
# --------------------------------------------------
labels = label(mask)
if labels.max() > 0:
    largest_cc = labels == (np.bincount(labels.flat)[1:].argmax() + 1)
else:
    largest_cc = mask.copy()

# --------------------------------------------------
# Napari visualization
# --------------------------------------------------
viewer.add_image(mask.astype(np.uint8), name="original", blending="additive")

viewer.add_image(filled_3d.astype(np.uint8), name="filled_3d")
viewer.add_image(filled_closed_3d.astype(np.uint8), name="closed+filled_3d")
viewer.add_image(holes_removed_3d.astype(np.uint8), name="remove_small_holes_3d")

viewer.add_image(filled_2d.astype(np.uint8), name="filled_2d_slice")
viewer.add_image(filled_2d_closed.astype(np.uint8), name="closed+filled_2d_slice")

viewer.add_image(largest_cc.astype(np.uint8), name="largest_component")

<Image layer 'largest_component [2]' at 0x1e7bd2a34c0>

In [ ]:
nuclei_mask = nuclei_labels > 0
dist = distance_transform_edt(~nuclei_mask)

In [ ]:
dist_mask = dist < 15
viewer.add_image(dist_mask, name="distance_transform", colormap="gray", blending="additive", opacity=0.5)

In [ ]:
from scipy.ndimage import binary_fill_holes

# Apply binary closing

# Fill holes in the mask
root_mask = binary_fill_holes(root_mask)
viewer.add_image(root_mask, name="root_mask", colormap="gray", blending="additive", opacity=0.5)

In [ ]:
from skimage.morphology import binary_erosion, ball

# Perform binary erosion of root_mask by 2 pixels
eroded_root_mask = binary_erosion(root_mask, footprint=ball(2))
viewer.add_image(eroded_root_mask, name="root_mask_eroded", colormap="gray", blending="additive", opacity=0.5)

In [ ]:
from scipy.ndimage import distance_transform_edt
import numpy as np

# Compute the Euclidean distance from each nuclei_label pixel to the nearest background (0-valued) pixel in eroded_root_mask
distance_to_bg = distance_transform_edt(nuclei_labels == eroded_root_mask[0])

# Prepare an empty array to hold the per-nucleus distance values
nuclei_label_distances = np.zeros_like(nuclei_labels, dtype=distance_to_bg.dtype)

# Assign the minimum distance value inside each nucleus label to all its pixels
for label_id in np.unique(nuclei_labels):
    if label_id == 0:
        continue  # Skip background
    mask = nuclei_labels == label_id
    min_distance = distance_to_bg[mask].min() if np.any(mask) else 0
    nuclei_label_distances[mask] = min_distance

viewer.add_image(nuclei_label_distances, name="nuclei_dist_to_bg", colormap="viridis", blending="additive", opacity=0.6)

In [ ]:
viewer.add_image(nuclei_label_distances, name="nuclei_dist_to_bg", colormap="viridis", blending="additive", opacity=0.6)

In [ ]:
bf_inv.shape